# Scaling the Investment Advisor

In the previous notebook we built a 4-agent Investment Advisor for a single stock. It works — but it has a key limitation you'd hit immediately in production:

| Section | The Limitation | The Fix | What It Enables |
|---------|---------------|---------|------------------|
| 1 | Can only analyze one stock at a time | **Async Multi-Stock** | Analyze a watchlist of 3 stocks simultaneously |

In [1]:
import json
import os
import time
from typing import Any, Literal, Tuple

import yfinance as yf
from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from crewai.tools import tool
from crewai_tools import EXASearchTool
from curl_cffi import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

/Users/ishandutta/miniconda3/envs/agents/lib/python3.10/site-packages/crewai_tools/tools/exa_tools/exa_search_tool.py:40: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  api_key: Optional[str] = Field(


First, let's set up the tools and agents from v1 so we have a working baseline.

In [2]:
session = requests.Session(impersonate="chrome")

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")
exa_search_tool = EXASearchTool()


@tool("Get current stock price")
def get_current_stock_price(symbol: str) -> str:
    """Get the current stock price for a given symbol."""
    try:
        time.sleep(0.5)
        stock = yf.Ticker(symbol, session=session)
        current_price = stock.info.get(
            "regularMarketPrice", stock.info.get("currentPrice")
        )
        return (
            f"{current_price:.2f}"
            if current_price
            else f"Could not fetch current price for {symbol}"
        )
    except Exception as e:
        return f"Error fetching current price for {symbol}: {e}"


@tool
def get_company_info(symbol: str):
    """Get company information and current financial snapshot for a given stock symbol."""
    try:
        company_info_full = yf.Ticker(symbol, session=session).info
        if company_info_full is None:
            return f"Could not fetch company info for {symbol}"
        company_info_cleaned = {
            "Name": company_info_full.get("shortName"),
            "Symbol": company_info_full.get("symbol"),
            "Current Stock Price": f"{company_info_full.get('regularMarketPrice', company_info_full.get('currentPrice'))} {company_info_full.get('currency', 'USD')}",
            "Market Cap": f"{company_info_full.get('marketCap', company_info_full.get('enterpriseValue'))} {company_info_full.get('currency', 'USD')}",
            "Sector": company_info_full.get("sector"),
            "Industry": company_info_full.get("industry"),
            "Country": company_info_full.get("country"),
            "EPS": company_info_full.get("trailingEps"),
            "P/E Ratio": company_info_full.get("trailingPE"),
            "52 Week Low": company_info_full.get("fiftyTwoWeekLow"),
            "52 Week High": company_info_full.get("fiftyTwoWeekHigh"),
            "Revenue Growth": company_info_full.get("revenueGrowth"),
            "Gross Margins": company_info_full.get("grossMargins"),
            "EBITDA": company_info_full.get("ebitda"),
        }
        return json.dumps(company_info_cleaned)
    except Exception as e:
        return f"Error fetching company profile for {symbol}: {e}"


@tool
def get_income_statements(symbol: str):
    """Get income statements for a given stock symbol."""
    try:
        stock = yf.Ticker(symbol, session=session)
        financials = stock.financials
        return financials.to_json(orient="index")
    except Exception as e:
        return f"Error fetching income statements for {symbol}: {e}"

In [3]:
class InvestmentRecommendation(BaseModel):
    action: Literal["BUY", "HOLD", "SELL"] = Field(description="Investment action")
    confidence: float = Field(description="Confidence score 0.0 to 1.0")
    target_price: float = Field(description="12-month target price")
    current_price: float = Field(description="Current stock price")
    reasons: list[str] = Field(description="Key reasons for recommendation")
    risks: list[str] = Field(description="Key risks to consider")


def validate_recommendation(result: TaskOutput) -> Tuple[bool, Any]:
    rec = result.pydantic
    if not rec or rec.confidence < 0 or rec.confidence > 1:
        return (False, "Confidence must be between 0.0 and 1.0")
    if len(rec.reasons) < 2:
        return (False, "Must provide at least 2 reasons")
    if len(rec.risks) < 1:
        return (False, "Must provide at least 1 risk")
    return (True, rec)

In [4]:
data_explorer = Agent(
    role="Data Researcher",
    goal="Gather and provide financial data and company information about a stock",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert researcher, who can gather detailed information about a company or stock. "
        'When using tools, use the stock symbol and add a suffix \".NS\" to it. '
        'Try with and without the suffix and see what works.'
    ),
    tools=[get_company_info, get_income_statements],
    max_iter=5,
    max_rpm=12,
    max_execution_time=450,
    respect_context_window=True,
)

news_info_explorer = Agent(
    role="News and Info Researcher",
    goal="Gather and provide the latest news and information about a company from the internet",
    llm=llm,
    verbose=True,
    backstory="You are an expert researcher, who can gather detailed information about a company.",
    tools=[exa_search_tool],
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

analyst = Agent(
    role="Data Analyst",
    goal="Consolidate financial data, stock information, and provide a summary",
    llm=llm,
    verbose=True,
    backstory=(
        "You are an expert in analyzing financial data, stock/company-related current information, and "
        "making a comprehensive analysis."
    ),
    max_iter=4,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

fin_expert = Agent(
    role="Financial Expert",
    goal="Considering financial analysis of a stock, make investment recommendations",
    backstory=(
        "You are an expert financial advisor who can provide investment recommendations. "
        "Consider the financial analysis, current information about the company, current stock price, "
        "and make recommendations about whether to buy/hold/sell a stock along with reasons. "
        'When using tools, try with and without the suffix \".NS\" to the stock symbol and see what works.'
    ),
    llm=llm,
    verbose=True,
    tools=[get_current_stock_price],
    max_iter=5,
    max_rpm=8,
    max_execution_time=360,
    respect_context_window=True,
)

print("Agents ready:", data_explorer.role, "/", news_info_explorer.role, "/", analyst.role, "/", fin_expert.role)

Agents ready: Data Researcher / News and Info Researcher / Data Analyst / Financial Expert


Now let's build the v1 crew that analyzes a single stock (our baseline for comparison).

In [5]:
def build_single_stock_crew():
    """Build the v1 crew for analyzing a single stock."""
    financials_task = Task(
        description="Get financial data like income statements and other fundamental ratios for stock: {stock}. "
        "Use the year 2026 as the current year.",
        expected_output="Detailed information from income statement, key ratios for {stock}.",
        agent=data_explorer,
    )

    news_task = Task(
        description="Get latest news and business information about company: {stock}. "
        "Use the year 2026 as the current year.",
        expected_output="Latest news and business information about the company.",
        agent=news_info_explorer,
    )

    analysis_task = Task(
        description="Make thorough analysis based on given financial data and latest news of a stock.",
        expected_output="Comprehensive analysis of a stock outlining financial health, stock valuation, risks, and news.",
        agent=analyst,
        context=[financials_task, news_task],
    )

    advise_task = Task(
        description="Make a recommendation about investing in a stock, based on analysis provided and current stock price.",
        expected_output="A structured investment recommendation with action, confidence, target price, reasons, and risks.",
        agent=fin_expert,
        context=[analysis_task],
        output_pydantic=InvestmentRecommendation,
        guardrail=validate_recommendation,
    )

    crew = Crew(
        agents=[data_explorer, news_info_explorer, analyst, fin_expert],
        tasks=[financials_task, news_task, analysis_task, advise_task],
        process=Process.sequential,
        verbose=True,
        cache=True,
        max_rpm=15,
    )
    return crew


print("build_single_stock_crew() ready")

build_single_stock_crew() ready


---

## Section 1: Async Multi-Stock Screener

### The Limitation

In v1, our advisor analyzes **one stock at a time**. If you want to compare 3 stocks (say AAPL, GOOGL, and MSFT), you'd run the crew 3 times sequentially — waiting for each to finish before starting the next.

```
v1 approach (sequential):

AAPL ████████████████  (5 min)
                      GOOGL ████████████████  (5 min)
                                              MSFT ████████████████  (5 min)
                                                                     = 15 min total
```

### The Fix: `kickoff_async()` + `asyncio.gather()`

CrewAI has a native async method called `kickoff_async()`. Combined with Python's `asyncio.gather()`, we can run all 3 analyses **at the same time**.

```
v2 approach (async parallel):

AAPL  ████████████████
GOOGL ████████████████   (all 3 running simultaneously)
MSFT  ████████████████
                        = ~5 min total
```

### Before: One stock at a time

This is what you'd do without async — a simple loop. Each stock waits for the previous one to finish.

In [ ]:
watchlist = ["AAPL", "GOOGL", "MSFT"]

# sequential_start = time.time()
# sequential_results = {}

# for stock in watchlist:
#     crew = build_single_stock_crew()
#     result = crew.kickoff(inputs={"stock": stock})
#     sequential_results[stock] = result.pydantic
#     print(f"{stock}: {result.pydantic.action} (confidence: {result.pydantic.confidence})")

# sequential_time = time.time() - sequential_start
# print(f"\nSequential total: {sequential_time:.2f}s ({sequential_time/60:.1f} min)")

### After: All stocks at once

With `kickoff_async()`, each crew runs as an async task. `asyncio.gather()` starts them all at the same time and waits for all to finish.

- `kickoff()` — synchronous, blocks until done
- `kickoff_async()` — async, returns immediately, runs in the background

In [6]:
import asyncio


async def analyze_watchlist(stocks: list[str]):
    crews_and_stocks = []
    for stock in stocks:
        crew = build_single_stock_crew()
        crews_and_stocks.append((crew, stock))

    tasks = [
        crew.kickoff_async(inputs={"stock": stock})
        for crew, stock in crews_and_stocks
    ]

    results = await asyncio.gather(*tasks)
    return dict(zip(stocks, results))


watchlist = ["AAPL", "GOOGL", "MSFT"]

async_start = time.time()
async_results = await analyze_watchlist(watchlist)
async_time = time.time() - async_start

for stock, result in async_results.items():
    rec = result.pydantic
    print(f"{stock}: {rec.action} (confidence: {rec.confidence})")

print(f"\nAsync total: {async_time:.2f}s ({async_time/60:.1f} min)")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3eb3cb33-b191-4fcf-b9a3-b04c929eb3a6                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d7a50626-e04c-4b38-9d04-63e245567a96                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dd1da0d9-95a0-4993-9d03-22f83f4fb79e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│  Task: Get financial data like income statements and other fundamental ratios for stock: AAPL. Use the year     │
│  2026 as the current year.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_recommendation(result: TaskOutput) ->...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: de843f77-a24c-4d00-ab9d-4a2d6a76cf13                                                                     │
│  Agent: Financial Expert                                                                                        │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dd1da0d9-95a0-4993-9d03-22f83f4fb79e                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "action": "Buy",                                                                                             │
│    "confidence": 0.85,                                                                                          │
│    "target_price": 520.00,                                                                                      │
│    "current_price": 424.62,                                                                                     │
│    "reasons": [                                                                                                 │
│      "Consistently strong revenue and earnings growth with robust margins; most recent quarter shows            │
│  accelerating profit expansion.",                                                                               │
│      "Cloud and AI are now the main drivers, with Azure and AI-powered Copilot delivering rapid, high-margin    │
│  growth.",                                                                                                      │
│      "High recurring revenues, dominant competitive positioning, and substantial reinvestment in R&D enable     │
│  Microsoft to maintain innovation leadership and adaptability.",                                                │
│      "Valuation (P/E 26.6) is reasonable given ~20%+ EPS growth, sector leadership, and historical premium      │
│  compared to peers.",                                                                                           │
│      "Balance sheet strength and exceptional free cash flow support continued shareholder returns               │
│  (buybacks/dividends) and risk mitigation."                                                                     │
│    ],                                                                                                           │
│    "risks": [                                                                                                   │
│      "Stock trades at a growth-premium valuation, so multiple contraction risk exists if there is any slowdown  │
│  or broad market tech selloff.",                                                                                │
│      "Cloud and AI remain highly competitive with AWS and Google, while legacy business transitions still       │
│  matter long-term.",                                                                                            │
│      "Ongoing global regulatory scrutiny, especially for competition/antitrust in AI and cloud, could bring     │
│  headline or structural risk.",                                                                                 │
│      "Macro/geopolitical risks (FX movements, global demand shifts, U.S.-China tensions) can produce            │
│  meaningful short-term volatility."                                                                             │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                       

AAPL: HOLD (confidence: 0.8)
GOOGL: BUY (confidence: 0.88)
MSFT: BUY (confidence: 0.85)

Async total: 57.14s (1.0 min)


### Comparing results across stocks

With all results available at once, we can compare them side-by-side.

In [7]:
print(f"{'Stock':<8} {'Action':<6} {'Confidence':<12} {'Current':<10} {'Target':<10}")
print("-" * 50)

for stock, result in async_results.items():
    rec = result.pydantic
    print(f"{stock:<8} {rec.action:<6} {rec.confidence:<12.2f} ${rec.current_price:<9.2f} ${rec.target_price:<9.2f}")

buy_stocks = [s for s, r in async_results.items() if r.pydantic.action == "BUY"]
print(f"\nStocks to BUY: {buy_stocks if buy_stocks else 'None'}")

best = max(async_results.items(), key=lambda x: x[1].pydantic.confidence)
print(f"Highest confidence: {best[0]} ({best[1].pydantic.confidence:.0%})")

Stock    Action Confidence   Current    Target    
--------------------------------------------------
AAPL     HOLD   0.80         $271.06    $310.00   
GOOGL    BUY    0.88         $344.40    $390.00   
MSFT     BUY    0.85         $424.62    $520.00   

Stocks to BUY: ['GOOGL', 'MSFT']
Highest confidence: GOOGL (88%)


With async, we went from analyzing stocks one at a time to analyzing an entire watchlist simultaneously. Now let's wrap this into a complete portfolio screener pipeline.

---

## Recap

| Section | You Learned | CrewAI API |
|---------|------------|------------|
| 1 | Analyze multiple stocks at once | `crew.kickoff_async()` + `asyncio.gather()` |

You started with a single-stock advisor and ended with a portfolio screener that analyzes multiple stocks simultaneously and presents a unified comparison.